# 08 — Self-improvement loop

Demonstrates how user feedback flows into:
1. **Chunk-quality scores** (re-ranking)
2. **Learned rules** (added to system prompt)
3. **Golden Q&A pairs** (few-shot examples)

Run notebooks 06 and 07 first so there's some history & sources to reference.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.cosmos_client import create_state_service
from src.openai_client import OpenAIService
from src.learning import LearningLoop
from src.models import FeedbackRecord

# Returns CosmosService if configured, else LocalStateService (file-backed JSON).
cosmos = create_state_service(); cosmos.ensure_containers()
print('State backend:', type(cosmos).__name__)
ai = OpenAIService()
learner = LearningLoop(cosmos, ai)

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: LocalStateService ready: ['sessions', 'documents', 'feedback', 'learned_rules', 'golden_pairs', 'chunk_quality', 'ingestion_tasks']


State backend: LocalStateService


In [2]:
# Simulate three corrections (👎 with text)
for question, bad, fix in [
    ('How many nodes in the cluster?', '5', 'Actually 3 — the diagram on page 4 shows 3 nodes.'),
    ('What is the SLA?',                '99.9%', 'It is 99.95% per the SRE doc.'),
    ('When was the report published?',  '2024',  'It was published in Q3 2025.'),
]:
    cosmos.save_feedback(FeedbackRecord(
        session_id='nb-learn', turn_id='dummy', rating='down',
        correction=fix, question=question, answer=bad,
        chunk_ids=['chunk-x']
    ))

# Simulate two thumbs-ups
for question, good in [
    ('What is DocMind?',  'A self-improving multimodal RAG agent.'),
    ('Where is the SLA defined?', 'On page 12 of the SRE document.'),
]:
    cosmos.save_feedback(FeedbackRecord(
        session_id='nb-learn', turn_id='dummy', rating='up',
        question=question, answer=good, chunk_ids=['chunk-y']
    ))
print('feedback seeded ✔')

feedback seeded ✔


In [3]:
stats = learner.run_once()
print('Learning stats:', stats)
print('\nRules learned:')
for r in cosmos.get_rules():
    print(' •', r.rule)
print('\nGolden pairs:')
for g in cosmos.get_golden_pairs():
    print(' Q:', g.question, '→', g.answer[:80])

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Learning loop done: {'feedback_count': 10, 'rules_added': 4, 'golden_added': 4, 'chunk_updates': 10}


Learning stats: {'feedback_count': 10, 'rules_added': 4, 'golden_added': 4, 'chunk_updates': 10}

Rules learned:
 • Verify the accuracy of numerical data before responding.
 • Reference specific documents or sources when providing information.
 • Ensure consistency in answers for repeated questions.
 • Cross-check information with diagrams or visual aids when available.
 • Verify the accuracy of the information before responding.

Golden pairs:
 Q: What is DocMind? → A self-improving multimodal RAG agent.
 Q: Where is the SLA defined? → On page 12 of the SRE document.
 Q: What is DocMind? → A self-improving multimodal RAG agent.


In [4]:
# Verify chunk-quality scoring
print('chunk-x:', cosmos.get_chunk_quality('chunk-x'))
print('chunk-y:', cosmos.get_chunk_quality('chunk-y'))

chunk-x: id='chunk-x' chunk_id='chunk-x' times_retrieved=0 times_in_good_answer=0 times_in_bad_answer=3 quality_score=0.0 updated_at='2026-05-05T12:01:42.789474+00:00'
chunk-y: id='chunk-y' chunk_id='chunk-y' times_retrieved=0 times_in_good_answer=2 times_in_bad_answer=0 quality_score=1.0 updated_at='2026-05-05T12:01:42.739676+00:00'
